# Setup Folders

In [ ]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [ ]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

settings_path = "./settings.json"
data_folder = "./data"
os.makedirs(data_folder, exist_ok=True)

# Settings

In [ ]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

In [ ]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    OBSERVATION_CROP_DIM = settings["data_ingestion"]["observation_crop_dim"]
    OBSERVATION_DIM = settings["vae"]["model"]["observation_dim"]
    NUM_EXPLORATION_PROCESSES = settings["data_ingestion"]["num_exploration_processes"]
    TARGET_RUN_COUNT = settings["data_ingestion"]["target_run_count"]
    
def log_settings():
    logger.info(f"OBSERVATION_CROP_DIM: {OBSERVATION_CROP_DIM}")
    logger.info(f"OBSERVATION_DIM: {OBSERVATION_DIM}")
    logger.info(f"NUM_EXPLORATION_PROCESSES: {NUM_EXPLORATION_PROCESSES}")
    logger.info(f"TARGET_RUN_COUNT: {TARGET_RUN_COUNT}")

log_settings()

# Install and Load Packages

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import itertools
import json
import multiprocessing
import os
from typing import Dict, Any

from tqdm.notebook import tqdm

from src.car_racing_worker import run_single_exploration

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning" # Pygame has a deprecation warning at the moment

# Setup

In [ ]:
def ensure_count_file_exists(data_folder):
  run_count_file_path = os.path.join(data_folder, "run_count.txt")
  if not os.path.exists(run_count_file_path):
    with open(run_count_file_path, 'w') as count_file:
      count = str(0).zfill(7)
      count_file.write(str(count))


In [ ]:
ensure_count_file_exists(data_folder)

# Functions

In [ ]:
def get_current_count(data_folder):
  run_count_file_path = os.path.join(data_folder, "run_count.txt")
  try:
    with open(run_count_file_path, "r") as count_file:
      count_str = count_file.read().strip()
      if count_str:
        return int(count_str)
      return 0
  except (FileNotFoundError, ValueError):
    return 0

In [ ]:
def load_metadata(data_folder: str) -> Dict[str, Dict[str, Any]]:
    metadata = {}
    metadata_path = os.path.join(data_folder, "metadata.json")
    if os.path.exists(metadata_path):
        with open(metadata_path, "r") as file:
            metadata = json.load(file)
    return metadata

def save_metadata(metadata: Dict[str, Dict[str, Any]], data_folder: str) -> None:
    metadata_path = os.path.join(data_folder, "metadata.json")
    with open(metadata_path, "w") as file:
        json.dump(metadata, file)

In [ ]:
def run_multiple_explorations(data_folder, num_processes, target_count, logger):
    logger.info(f"Starting {num_processes} parallel processes")
    start_count = get_current_count(data_folder)
    if start_count > target_count:
        logger.warning(f"Current next exploration ({start_count}) already meets or exceeds target ({target_count})")
        return
    
    metadata = load_metadata(data_folder)
    ctx = multiprocessing.get_context('spawn')
    with ctx.Pool(processes=num_processes) as pool:
        kwargs = {
            "data_folder": data_folder,
            "y_crop_dim": OBSERVATION_CROP_DIM,
            "observation_dim": OBSERVATION_DIM,
            "logger": logger
        }
        tasks_iterable = itertools.repeat(kwargs)
        with tqdm(total=target_count, desc="Explorations Completed") as pbar:
            pbar.n = start_count
            pbar.refresh()
            result_pool = pool.imap_unordered(run_single_exploration, tasks_iterable)
            for result_count, result_file_name, result_length in result_pool:
                logger.debug(result_count)
                logger.debug(result_file_name)
                logger.debug(result_length)
                metadata[result_file_name] = {"length": result_length}
                if result_count >= target_count:
                    logger.debug(f"\nReached target count {result_count}")
                    logger.debug("Terminating all worker processes...")
                    pool.terminate()
                    pool.join()
                    logger.info(f"Finished all {target_count} explorations")
                    logger.debug(metadata)
                    save_metadata(metadata, data_folder)
                    return
                pbar.n = result_count + 1
                pbar.refresh()

# Run

In [ ]:
run_multiple_explorations(data_folder,
                          num_processes=NUM_EXPLORATION_PROCESSES,
                          target_count=TARGET_RUN_COUNT,
                          logger=logger)

# Upload

In [ ]:
if is_environment_colab_notebook():
    remote_folder = get_secret("worldModelsFolderPath")
    !tar -cf data.tar ./data
    !cp ./data.tar "$remote_folder"